In [1]:
import os, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import math

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByClass: 814,255 chars, 62 UNBALANCED classes
#  label mapping (0-indexed): 0-9 digits, 10-35 A-Z, 36-61 a-z
#  Hardest EMNIST split: uppercase and lowercase are SEPARATE classes,
#  so 'A'≠'a'. Key confusions: c/C, o/O, p/P, u/U, v/V, w/W, x/X.
#  Severe imbalance — macro F1 is the primary metric, not accuracy.
# ─────────────────────────────────────────────────────────────────────────────
NUM_CLASSES  = 62
IMG          = 28
BS           = 64
EPOCHS       = 100
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/byclass"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELS = (
    [str(d) for d in range(10)]
    + [chr(c) for c in range(65, 91)]   # A-Z
    + [chr(c) for c in range(97, 123)]  # a-z
)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"[INFO] Using device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
# EMNIST images are transposed/flipped in torchvision — apply fix transform
fix_transform = transforms.Lambda(lambda img: transforms.functional.rotate(
    transforms.functional.hflip(img), -90))

train_transform = transforms.Compose([
    fix_transform,
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.20, contrast=0.20),
    transforms.Pad(2, fill=-1),
    transforms.RandomCrop(IMG),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # maps [0,1] → [-1, 1]
])

val_transform = transforms.Compose([
    fix_transform,
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

print("[INFO] Loading EMNIST ByClass ...")
full_train = datasets.EMNIST(root="./data", split="byclass", train=True,
                              download=True, transform=train_transform)
test_ds    = datasets.EMNIST(root="./data", split="byclass", train=False,
                              download=True, transform=val_transform)

total   = len(full_train)
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val

# Split — keep val set with val_transform (no augmentation)
train_idx = list(range(n_train))
val_idx   = list(range(n_train, total))

train_ds_aug = Subset(full_train, train_idx)

# Build a clean copy with val_transform for the val split
full_val = datasets.EMNIST(root="./data", split="byclass", train=True,
                            download=False, transform=val_transform)
val_ds   = Subset(full_val, val_idx)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds):,}")

# ─────────────────────────────────────────────────────────────────────────────
#  CLASS WEIGHTS  —  counter severe imbalance
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Computing class weights ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in full_train:      # uses the already-loaded dataset
    label_counts[int(label)] += 1
label_counts = label_counts[:n_train]   # approximate from full — fast

# Recount properly from targets array (torchvision stores them)
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
targets = np.array(full_train.targets)
for c in range(NUM_CLASSES):
    label_counts[c] = int((targets[train_idx] == c).sum())

total_samples = label_counts.sum()
class_weights = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights = np.clip(class_weights, 0.1, 10.0)
class_weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

train_loader = DataLoader(train_ds_aug, batch_size=BS, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,       batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,      batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, max(1, channels // reduction)),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction), channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        attn = self.fc(self.gap(x)).view(x.size(0), -1, 1, 1)
        return x * attn


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return F.gelu(self.net(x) + x)


class DenseResBlock(nn.Module):
    """3 residual blocks with dense concat → depthwise-stride downsample."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = (
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            )
            if in_ch != out_ch else nn.Identity()
        )
        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)
        # concat of r1+r2+r3 → out_ch, then stride-2 dw conv
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch * 3, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.down = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                      groups=out_ch, bias=False),
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        x  = self.proj(x)
        r1 = self.r1(x)
        r2 = self.r2(r1)
        r3 = self.r3(r2)
        out = self.fuse(torch.cat([r1, r2, r3], dim=1))
        return self.down(out)


class AdaptiveFilterCapsule(nn.Module):
    """Lightweight capsule routing for num_classes capsules."""

    def __init__(self, in_dim, num_classes, capsule_dim=32):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_dim, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.x_proj = nn.Linear(in_dim, capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        h = F.gelu(self.fc1(x))
        h = self.fc2(h).view(-1, self.num_classes, self.capsule_dim)  # (B, K, D)
        x_s = self.x_proj(x).unsqueeze(1).expand(-1, self.num_classes, -1)  # (B, K, D)
        caps = (x_s * h).sum(dim=-1)   # (B, K)
        return self.bn(caps)


# ─────────────────────────────────────────────────────────────────────────────
#  MODEL  —  4-stage encoder for 62-class depth requirement
# ─────────────────────────────────────────────────────────────────────────────
class WhatNetByClass(nn.Module):
    def __init__(self, num_classes=62, image_size=28):
        super().__init__()
        K = num_classes

        # ── Stem ──────────────────────────────────────────────────────────
        self.stem_3x3 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU())
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 16, (1, 5), padding=(0, 2), bias=False), nn.BatchNorm2d(16), nn.GELU())
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 16, (5, 1), padding=(2, 0), bias=False), nn.BatchNorm2d(16), nn.GELU())
        self.stem_ca   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64), nn.GELU())

        # ── Encoder ───────────────────────────────────────────────────────
        self.enc1 = DenseResBlock(64,  64)   # 28→14
        self.enc2 = DenseResBlock(64,  128)  # 14→7
        self.enc3 = DenseResBlock(128, 256)  # 7→4
        self.enc4 = DenseResBlock(256, 512)  # 4→2

        # ── Multi-scale GAP projections ───────────────────────────────────
        self.gap1_proj = nn.Linear(64,  128)
        self.gap2_proj = nn.Linear(128, 128)
        self.gap3_proj = nn.Linear(256, 128)
        # gap4: 512 → kept raw; fused = 128*3 + 512 = 896

        # ── Capsule ───────────────────────────────────────────────────────
        self.afc = AdaptiveFilterCapsule(896, K, capsule_dim=32)

        # ── Dense head ────────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(896, 256, bias=False),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
        )
        self.head_logits = nn.Linear(256, K)

        # ── Gated fusion ──────────────────────────────────────────────────
        self.gate = nn.Linear(K * 2, 2)

    def forward(self, x):
        # Stem
        t    = self.stem_3x3(x)
        h    = self.stem_h(x)
        v    = self.stem_v(x)
        stem = self.stem_ca(torch.cat([t, h, v], dim=1))
        stem = self.stem_proj(stem)

        # Encoder
        e1 = self.enc1(stem)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # Multi-scale GAP
        def gap(t): return t.mean(dim=[2, 3])
        g1 = F.gelu(self.gap1_proj(gap(e1)))
        g2 = F.gelu(self.gap2_proj(gap(e2)))
        g3 = F.gelu(self.gap3_proj(gap(e3)))
        g4 = gap(e4)
        fused = torch.cat([g1, g2, g3, g4], dim=1)  # (B, 896)

        # Capsule branch
        afc_out = self.afc(fused)                    # (B, K)

        # Dense head branch
        x_head   = self.head(fused)
        x_logits = self.head_logits(x_head)          # (B, K)

        # Gated fusion
        gate  = F.softmax(self.gate(torch.cat([x_logits, afc_out], dim=1)), dim=1)
        out   = x_logits * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return out


# ─────────────────────────────────────────────────────────────────────────────
#  LABEL-SMOOTHED CROSS-ENTROPY (weighted)
# ─────────────────────────────────────────────────────────────────────────────
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.weight    = weight   # (num_classes,) tensor on device

    def forward(self, logits, targets):
        # targets: (B,) integer class indices
        log_prob = F.log_softmax(logits, dim=-1)           # (B, K)
        K        = logits.size(-1)
        smooth   = self.smoothing / K
        nll      = -log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)

        if self.weight is not None:
            w   = self.weight[targets]
            nll = nll * w

        uniform  = -log_prob.mean(dim=-1)
        loss     = (1.0 - self.smoothing) * nll + self.smoothing * uniform

        if self.weight is not None:
            return loss.sum() / w.sum()
        return loss.mean()


# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE — warmup + cosine decay
# ─────────────────────────────────────────────────────────────────────────────
def warmup_cosine_schedule(optimizer, total_steps, warmup_steps, base_lr):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(0.5 * (1.0 + math.cos(math.pi * progress)), 1e-6 / base_lr)
    return LambdaLR(optimizer, lr_lambda)


# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN + EVAL HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0; correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits  = model(images)
        loss    = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def compute_metrics(model, loader, num_classes, device):
    model.eval()
    tp = np.zeros(num_classes); fp = np.zeros(num_classes); fn = np.zeros(num_classes)
    correct_per_class = np.zeros(num_classes)
    total_per_class   = np.zeros(num_classes)
    for images, labels in loader:
        images  = images.to(device)
        preds   = model(images).argmax(1).cpu().numpy()
        lbls    = labels.numpy()
        for c in range(num_classes):
            tp[c] += ((preds == c) & (lbls == c)).sum()
            fp[c] += ((preds == c) & (lbls != c)).sum()
            fn[c] += ((preds != c) & (lbls == c)).sum()
            correct_per_class[c] += (preds[lbls == c] == c).sum()
            total_per_class[c]   += (lbls == c).sum()
    prec     = tp / (tp + fp + 1e-8)
    rec      = tp / (tp + fn + 1e-8)
    f1       = 2 * prec * rec / (prec + rec + 1e-8)
    macro_f1 = float(f1.mean() * 100.0)
    per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
    return macro_f1, per_class_acc


# ─────────────────────────────────────────────────────────────────────────────
#  MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
model     = WhatNetByClass(NUM_CLASSES, IMG).to(DEVICE)
criterion = LabelSmoothingCE(smoothing=LABEL_SMOOTH, weight=class_weight_tensor)
val_crit  = LabelSmoothingCE(smoothing=0.0)   # no smoothing for val loss display

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[INFO] Trainable params: {total_params:,}")

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
warmup_steps = steps_per_epoch * WARMUP_EP
scheduler    = warmup_cosine_schedule(optimizer, total_steps, warmup_steps, LR)

best_val_acc = 0.0
patience_ctr = 0
PATIENCE     = 12
history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, val_crit, DEVICE)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:>3}/{EPOCHS} | "
          f"train_loss: {train_loss:.4f}  train_acc: {train_acc*100:.2f}%  "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(),
                   os.path.join(RESULTS_DIR, "best_model.pt"))
        print(f"         → New best val_acc: {best_val_acc*100:.2f}%  (saved)")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"[INFO] Early stopping at epoch {epoch} (patience={PATIENCE})")
            break

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE on test set
# ─────────────────────────────────────────────────────────────────────────────
# Restore best weights
model.load_state_dict(torch.load(
    os.path.join(RESULTS_DIR, "best_model.pt"), map_location=DEVICE))

test_loss, test_acc = evaluate(model, test_loader, val_crit, DEVICE)
test_acc_pct = test_acc * 100.0

macro_f1, per_class_acc = compute_metrics(model, test_loader, NUM_CLASSES, DEVICE)

worst5_idx = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc_pct:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {total_params:,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYCLASS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc_pct, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": total_params,
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

print(f"\n[INFO] Saved to {RESULTS_DIR}/")
print("[DONE]")

[INFO] Using device: cuda
[INFO] Loading EMNIST ByClass ...
[INFO] Train: 628,139 | Val: 69,793 | Test: 116,323
[INFO] Computing class weights ...
[INFO] Class weight range: [0.29, 5.90]
[INFO] Steps/epoch: 9814
[INFO] Trainable params: 21,470,180
Epoch   1/100 | train_loss: 2.1367  train_acc: 52.87%  val_loss: 0.9589  val_acc: 74.86%
         → New best val_acc: 74.86%  (saved)
Epoch   2/100 | train_loss: 1.4147  train_acc: 74.39%  val_loss: 0.7538  val_acc: 78.88%
         → New best val_acc: 78.88%  (saved)
Epoch   3/100 | train_loss: 1.3300  train_acc: 76.75%  val_loss: 0.7378  val_acc: 75.54%
Epoch   4/100 | train_loss: 1.2859  train_acc: 77.77%  val_loss: 0.7098  val_acc: 77.40%
Epoch   5/100 | train_loss: 1.2555  train_acc: 78.62%  val_loss: 0.6816  val_acc: 79.61%
         → New best val_acc: 79.61%  (saved)
Epoch   6/100 | train_loss: 1.2296  train_acc: 79.25%  val_loss: 0.7095  val_acc: 77.05%
Epoch   7/100 | train_loss: 1.2060  train_acc: 79.97%  val_loss: 0.6569  val_acc: 7